# 🏛️ Servidor de Clasificación de Trámites por IA - Municipalidad de La Victoria

Este notebook de Google Colab aloja un modelo avanzado de Procesamiento de Lenguaje Natural (PLN) para la **clasificación de trámites y solicitudes en tiempo real**.

### 🧠 ¿Qué modelo usamos?
Utilizamos **`MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`**, un modelo multilingüe de Hugging Face de última generación optimizado para **clasificación Zero-shot** en español. A diferencia de modelos simples, este modelo puede clasificar texto con alta precisión en cualquier categoría definida dinámicamente sin necesidad de entrenamiento previo.

### ⚡ Funcionalidades:
1. Clasificación automática de **Prioridad** (Alta, Media, Baja).
2. Identificación del **Área Destino** (Acción sugerida).
3. Cálculo del **Porcentaje de certeza (confianza)**.
4. Exposición de un API público a través de un túnel seguro.

## 🛠️ Paso 1: Instalar dependencias e importar librerías

In [ ]:
# Instalar las dependencias necesarias
!pip install -q transformers[sentencepiece] fastapi uvicorn nest_asyncio pyngrok torch

In [ ]:
import torch
from transformers import pipeline
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import nest_asyncio
import uvicorn
import threading

# Verificar si tenemos GPU disponible en Colab para mayor velocidad
device = 0 if torch.cuda.is_available() else -1
print(f"🖥️ Dispositivo de procesamiento activo: {'GPU (CUDA)' if device == 0 else 'CPU'}")

## 🧠 Paso 2: Cargar el modelo Transformer (mDeBERTa-v3)

In [ ]:
print("📥 Descargando y cargando el modelo en memoria (esto puede tomar un minuto)...")

# Inicializamos el pipeline de clasificación zero-shot
classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device=device
)

print("✅ ¡Modelo cargado correctamente y listo para clasificar!")

## 🧪 Paso 3: Probar el modelo localmente
Hagamos una prueba rápida para comprobar la precisión del modelo en español.

In [ ]:
def clasificar_texto_local(texto):
    # 1. Definimos etiquetas de prioridad y ejecutamos clasificación
    # Usamos etiquetas claras y una plantilla descriptiva
    resultado_prioridad = classifier(
        texto,
        candidate_labels=["Alta", "Media", "Baja"],
        hypothesis_template="La urgencia e importancia de este reporte ciudadano es {}."
    )
    prioridad = resultado_prioridad["labels"][0]
    confianza_prioridad = resultado_prioridad["scores"][0]

    # 2. Definimos etiquetas de área y ejecutamos clasificación
    # Hacer las etiquetas más descriptivas ayuda enormemente al modelo zero-shot a contextualizar
    areas_mapeo = {
        "Defensa Civil (Emergencias y Peligros de cables, postes, derrumbes u objetos cayendo)": "Defensa Civil",
        "Saneamiento, Obras Públicas e Infraestructura de pistas, veredas o alcantarillado": "Saneamiento y Obras Públicas",
        "Desarrollo Económico, Licencias y Comercio": "Desarrollo Económico y Licencias",
        "Seguridad Ciudadana y Serenazgo": "Seguridad Ciudadana",
        "Fiscalización, Control y Multas": "Fiscalización y Control",
        "Mesa de Partes (Trámite Administrativo General y Documentos)": "Mesa de Partes / Trámite General"
    }
    
    resultado_area = classifier(
        texto,
        candidate_labels=list(areas_mapeo.keys()),
        hypothesis_template="El departamento municipal responsable de resolver este tipo de problema es {}."
    )
    area_descriptiva = resultado_area["labels"][0]
    confianza_area = resultado_area["scores"][0]
    area_sugerida = areas_mapeo[area_descriptiva]

    # Calculamos la certeza ponderando ambas clasificaciones
    certeza = (confianza_prioridad + confianza_area) / 2

    return {
        "prioridad": prioridad,
        "certeza": round(certeza * 100, 2),
        "accion_sugerida": f"Derivar a {area_sugerida}"
    }

# Probar con un texto de ejemplo
prueba = "Hay un cable de alta tensión roto colgando cerca del colegio de la cuadra 5, es muy peligroso."
print("Texto:", prueba)
print("Resultado:", clasificar_texto_local(prueba))

## 🌐 Paso 4: Crear la API con FastAPI

In [ ]:
app = FastAPI(title="Servicio de Clasificación IA - La Victoria")

class TramitePayload(BaseModel):
    texto: str
    asunto: str = ""
    descripcion: str = ""

@app.get("/")
def home():
    return {"status": "online", "model": "mDeBERTa-v3-base-mnli-xnli", "version": "2.0.0"}

@app.post("/clasificar")
def clasificar_tramite(payload: TramitePayload):
    # Combinamos asunto y descripción si están presentes, o usamos el texto general
    texto_analizar = payload.texto
    if payload.asunto or payload.descripcion:
        texto_analizar = f"Asunto: {payload.asunto}\nDescripción: {payload.descripcion}"
    
    if not texto_analizar.strip():
        raise HTTPException(status_code=400, detail="El texto a clasificar no puede estar vacío.")
        
    try:
        resultado = clasificar_texto_local(texto_analizar)
        return resultado
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

## 🚀 Paso 5: Ejecutar Servidor y Exponer con Túnel

Puedes exponer este servidor a internet para conectarlo con tu backend utilizando **Ngrok** o **Localtunnel**. Elige una de las siguientes opciones:

### Opción A: Exponer usando LOCAL TUNNEL (Recomendado - 100% Gratis y sin registro)
Esta opción no requiere crear cuentas ni tokens. Solo ejecuta la celda y dale clic al enlace generado.

In [ ]:
# Habilitar nest_asyncio para poder correr uvicorn dentro del notebook
nest_asyncio.apply()

def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8000)

# Iniciar el servidor de FastAPI en un hilo secundario
threading.Thread(target=run_api, daemon=True).start()
print("🚀 Servidor FastAPI corriendo en http://127.0.0.1:8000")

# Exponer el puerto 8000 a internet mediante localtunnel
print("🌍 Creando túnel público con Localtunnel... (Copia la URL que termine en '.loca.lt')")
!npx localtunnel --port 8000

### Opción B: Exponer usando NGROK (Alternativa)
Si prefieres usar Ngrok, coloca tu Authtoken abajo y ejecuta la celda.

In [ ]:
from pyngrok import ngrok

# ⚠️ IMPORTANTE: Pega tu Authtoken de ngrok aquí (consíguelo gratis en dashboard.ngrok.com)
NGROK_TOKEN = "TU-NGROK-AUTHTOKEN-AQUI"

if NGROK_TOKEN == "TU-NGROK-AUTHTOKEN-AQUI":
    print("⚠️ Por favor ingresa tu token de ngrok para continuar o utiliza la opción A (Localtunnel).")
else:
    ngrok.set_auth_token(NGROK_TOKEN)
    
    # Abrir túnel HTTP en el puerto 8000
    public_url = ngrok.connect(8000)
    print(f"🌍 Túnel público ngrok activo en: {public_url}/clasificar")
    
    # Correr servidor
    nest_asyncio.apply()
    uvicorn.run(app, host="127.0.0.1", port=8000)